# Simple Notebook to run Quantum Volume Experiments

This notebook explores basic aspects of Quantum Volume, which can be used to quantify the computational power of a quantum device. It can be used to measure the Quantum Volume of a specific device or backend.

A width of n qubits passes the test — Quantum Volume 2ⁿ — if random "model circuits" of n qubits and depth n produce their *heavy outputs* (the more likely half of the outcomes) more than 2/3 of the time, with a confidence level above 97.7% (from at least 100 random circuits). The notebook uses the `QuantumVolume` experiment of [Qiskit Experiments](https://qiskit-community.github.io/qiskit-experiments/manuals/verification/quantum_volume.html), which also explains the method in detail.

By default it runs on a perfect simulator, so every width should pass. The test starts at 2 qubits. It uses 500 random circuits per width: with only 100, the 3-qubit test can fail even on a perfect simulator, because the confidence is too low.

Implemented by [Jan-R. Lahmann](https://github.com/JanLahmann) using Qiskit; updated for Qiskit 2.x.

In [ ]:
from qiskit_experiments.library import QuantumVolume
from qiskit_aer import AerSimulator
from qiskit.primitives import BackendSamplerV2 as Sampler  # local sampler, not IBM Runtime

# a perfect simulator
backend = AerSimulator(seed_simulator=42)

# or a noisy copy of a real IBM device (noise from a calibration snapshot); the test then runs on
# physical qubits 0, 1, …, which need not be the device's best (see GHZ-on-Real-Devices.ipynb)
# from qiskit_ibm_runtime.fake_provider import FakeTorino
# backend = AerSimulator.from_backend(FakeTorino(), seed_simulator=42)

sampler = Sampler(backend=backend)

In [ ]:
max_qubits = 5   # test the widths 2 … max_qubits
for n in range(2, max_qubits + 1):
    exp = QuantumVolume(tuple(range(n)), trials=500, seed=42)
    exp.set_transpile_options(optimization_level=3)
    data = exp.run(backend, sampler=sampler).block_for_results()
    results = data.analysis_results(dataframe=True).set_index('name')
    hop = results.loc['mean_HOP', 'value']
    verified = results.loc['quantum_volume', 'value'] == 2 ** n
    print(f"{n} qubits: heavy outputs {hop} of the time → Quantum Volume {2 ** n}",
          "verified" if verified else "NOT verified")